In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from typing import Dict, Tuple, Optional
import warnings

In [2]:
here = Path(__file__).resolve().parent if "__file__" in globals() else Path().resolve()
root = here

while not (root / "data").exists() and root != root.parent:
    root = root.parent

CHO = root / "data" / "CHO_GHThermo.xlsx"
CHOimp=pd.read_excel(CHO)

In [3]:
df=pd.DataFrame({
    "Formula": ["CO2", "NH3", "HNO3", "CH3O", "NH4NO3"]
})

def parse_formulas(formula): 
    while "(" in formula: 
        match =re.search(r"\(([A-Za-z0-9]+)\)(\d+)", formula)
        if not match: 
            break
        group, mult = match.groups()
        mult=int(mult)
        expanded="".join([group] * mult)
        formula=formula.replace(match.group(0), expanded, 1)
        
    matches=re.findall(r'([A-Z][a-z]*)(\d*)', formula)
    counts={}
    for elem, num in matches: 
        counts[elem] = counts.get(elem, 0) + int(num) if num else counts.get(elem, 0) + 1
    return counts

In [4]:
#Minimal Periodic Table
ATOMIC_WEIGHTS ={
    "H": 1.00794, "C": 12.011, "O": 15.999, "N": 14.007
}

def molar_mass(formula: str, weights: Optional[Dict[str, float]] = None) -> float: 
    """
    Compute molar mass (g/mol) using average atomic weights.
    """
    weights=weights or ATOMIC_WEIGHTS
    counts = parse_formulas(formula)
    mm = 0.0
    for el, qty in counts.items(): 
        if el not in weights: 
            raise KeyError(f"Atomic weight for element {el!r} not found.")
        mm += weights[el] * qty
    return mm

In [5]:
CHOdata=pd.DataFrame(CHOimp)
T0=298.15
CHOdata['dS_f (J/mol K)']=((CHOdata['dH_F (kJ/mol)']-CHOdata['dG_F (kJ/mol)'])/T0)*1000
print(CHOdata.to_string(index=False))

           Compound  Formula  dH_F (kJ/mol)  dG_F (kJ/mol)  dH_rxn (kJ/mol)  dG_rxn (kJ/mol)  dS_f (J/mol K)
            Methane      CH4         -74.87     -50.736845           890.30       817.903155      -80.943000
             Ethane     C2H6         -84.00     -32.090296          1560.51      1468.049704     -174.106000
            Propane     C3H8        -104.70     -24.107372          2219.15      2107.532628     -270.309000
             Butane    C4H10        -125.60     -16.288669          2877.59      2746.851331     -366.632000
            Pentane    C5H12        -146.80      -8.033536          3535.73      3386.606464     -465.425000
             Hexane    C6H14        -198.70      -3.799941          4163.17      4022.340059     -653.698000
            Heptane    C7H16        -224.40       1.508553          4816.81      4659.148553     -757.701000
             Octane    C8H18        -250.30       6.581270          5470.25      5295.721270     -861.584000
             Nonane

In [6]:
CHOdata["parsed"]=CHOdata["Formula"].apply(parse_formulas)
CHOdata["C"]=CHOdata["parsed"].apply(lambda x: x.get("C", 0))
CHOdata["H"]=CHOdata["parsed"].apply(lambda x: x.get("H", 0))
CHOdata["O"]=CHOdata["parsed"].apply(lambda x: x.get("O", 0))
CHOdata=CHOdata.drop(["parsed"], axis=1)
# print(CHOdata.to_string(index=False))

In [7]:
stdthermo=pd.DataFrame({
    "Compound": ["CO2", "H2O", "O2"], 
    "G_F (kJ/mol)": [-394.36, -237.14, 0], 
    "H_F (kJ/mol)": [-393.51, -285.83, 0]
})

In [8]:
entropy=pd.DataFrame({
    "Compound": ["C", "H2", "O2"], 
    "S0 (J/mol K)": [5.833, 130.68, 205.15]
})
elem=entropy.set_index("Compound")["S0 (J/mol K)"]
elemental_entropies=pd.Series({
    "C": elem["C"], 
    "H": elem["H2"]/2, 
    "O": elem["O2"]/2
})
CHOdata["S0 (J/molK)"]=CHOdata["dS_f (J/mol K)"] + (CHOdata[["C", "H", "O"]] @ elemental_entropies)
print(CHOdata.to_string(index=True))

               Compound   Formula  dH_F (kJ/mol)  dG_F (kJ/mol)  dH_rxn (kJ/mol)  dG_rxn (kJ/mol)  dS_f (J/mol K)   C   H  O  S0 (J/molK)
0               Methane       CH4         -74.87     -50.736845           890.30       817.903155      -80.943000   1   4  0   186.250000
1                Ethane      C2H6         -84.00     -32.090296          1560.51      1468.049704     -174.106000   2   6  0   229.600000
2               Propane      C3H8        -104.70     -24.107372          2219.15      2107.532628     -270.309000   3   8  0   269.910000
3                Butane     C4H10        -125.60     -16.288669          2877.59      2746.851331     -366.632000   4  10  0   310.100000
4               Pentane     C5H12        -146.80      -8.033536          3535.73      3386.606464     -465.425000   5  12  0   347.820000
5                Hexane     C6H14        -198.70      -3.799941          4163.17      4022.340059     -653.698000   6  14  0   296.060000
6               Heptane     C7H16 

In [9]:
CHOdata['vO2']=CHOdata['C']+CHOdata['H']/4-CHOdata['O']/2
x=CHOdata[['vO2']].values
y=CHOdata['dH_rxn (kJ/mol)'].values

model=LinearRegression().fit(x,y)
theta0=model.intercept_
kO2=model.coef_[0]
yhat = model.predict(x)

mae = mean_absolute_error(y, yhat)
mse = mean_squared_error(y, yhat)
rmse = np.sqrt(mse)
r2=r2_score(y, yhat)
eps = 1e-12
mape = np.mean(np.abs((y-yhat)/np.where(np.abs(y) < eps, np.nan, y))) * 100
medae = np.median(np.abs(y-yhat))

highlightKabo = ["a-D-Glucose", "Potato Starch"]
highlightIoe = ["C1", "C2", "C3", "C4", "CA"]
highlightDomal = ["Cellulose", "Lignin", "Xylan", "Hemicellulose", "Starch"]
highlight_set = set(highlightIoe + highlightKabo + highlightDomal)

fig, ax = plt.subplots(figsize=(8, 6), dpi=1200)

base_df = CHOdata[~CHOdata["Compound"].isin(highlight_set)]
ax.scatter(base_df['vO2'], base_df["dH_rxn (kJ/mol)"], s=40, color="tab:blue", alpha=0.95, label='NIST Data')
ax.plot(CHOdata['vO2'], yhat, color='r', label='Proposed Model')

ax.text(
    0.05, 0.95, 
    f"$H_C = {theta0:.2f} + {kO2:.2f}\\,\\nu_{{O_2}}$"
    "\n"
    f"$R^2 = {r2:.4f}$", 
    transform=ax.transAxes, va="top", fontsize=8, 
    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85)
)

sub = CHOdata[CHOdata["Compound"].isin(highlightIoe)].copy()
for k, (_, r) in enumerate(sub.iterrows()):
    ax.scatter(r["vO2"], r["dH_rxn (kJ/mol)"], 
               s=120, marker="^", facecolor="white", edgecolor="black", zorder=5, 
               label="Ioelovich" if k == 0 else "_nolegend_")

sub = CHOdata[CHOdata["Compound"].isin(highlightKabo)].copy()
for k, (_, r) in enumerate(sub.iterrows()):
    ax.scatter(r["vO2"], r["dH_rxn (kJ/mol)"], 
               s=120, marker="s", facecolor="white", edgecolor="black", zorder=5, 
               label="Kabo" if k == 0 else "_nolegend_")
    
sub = CHOdata[CHOdata["Compound"].isin(highlightDomal)].copy()
for k, (_, r) in enumerate(sub.iterrows()):
    ax.scatter(r["vO2"], r["dH_rxn (kJ/mol)"], 
               s=120, marker="*", facecolor="white", edgecolor="black", zorder=5, 
               label="Domalski" if k == 0 else "_nolegend_")

ax.set_xlabel(r"$\nu_{O_2}$ (mol $O_2$ (mol compound)$^{-1}$)")
ax.set_ylabel(r"$-\Delta H_{\text{C}}$ (kJ mol$^{-1}$)")
ax.legend(frameon=False)
ax.grid(False)

plt.tight_layout()
print(f"Theta0 = {theta0:.2f} kJ/mol, kO2 = {kO2:.2f} kJ per mol O2")
print(f"R2 = {r2:.4f}, MAE = {mae:.2f} kJ/mol, RMSE = {rmse:.2f} kJ/mol, "
      f"Median AE = {medae:.2f} kJ/mol, MAPE = {mape:.2f}%")
# plt.savefig("ModelDev.png", dpi=1200, bbox_inches="tight")
plt.show()
print(CHOdata.to_string(index=False))

Theta0 = 151.67 kJ/mol, kO2 = 422.84 kJ per mol O2
R2 = 0.9898, MAE = 78.82 kJ/mol, RMSE = 164.64 kJ/mol, Median AE = 52.62 kJ/mol, MAPE = 4.23%


           Compound  Formula  dH_F (kJ/mol)  dG_F (kJ/mol)  dH_rxn (kJ/mol)  dG_rxn (kJ/mol)  dS_f (J/mol K)  C  H  O  S0 (J/molK)  vO2
            Methane      CH4         -74.87     -50.736845           890.30       817.903155      -80.943000  1  4  0   186.250000  2.0
             Ethane     C2H6         -84.00     -32.090296          1560.51      1468.049704     -174.106000  2  6  0   229.600000  3.5
            Propane     C3H8        -104.70     -24.107372          2219.15      2107.532628     -270.309000  3  8  0   269.910000  5.0
             Butane    C4H10        -125.60     -16.288669          2877.59      2746.851331     -366.632000  4 10  0   310.100000  6.5
            Pentane    C5H12        -146.80      -8.033536          3535.73      3386.606464     -465.425000  5 12  0   347.820000  8.0
             Hexane    C6H14        -198.70      -3.799941          4163.17      4022.340059     -653.698000  6 14  0   296.060000  9.5
            Heptane    C7H16        -224.40     

In [10]:
kO2=422.84 #kJ/mol O2
theta0=151.67 #kJ/mol
T0 = 298.15 #K

def predict_org_thermo(
        formulas: str | list[str], 
        stdthermo: pd.DataFrame, 
        elemental_entropies: list[float]
) -> pd.DataFrame:
    """
    Estimate thermodynamic properties for a CxHyOz compound from molecular composition.

    Given a chemical formula, this model estimates: 
        - Enthalpy of combustion (ΔH_C)
        - Gibbs free energy of combustion (ΔG_C) 
        - Enthalpy of formation (ΔH_F)
        - Gibbs free energy of formation (ΔG_F)
        - Entropy of formation (ΔS_F)
        - Absolute entropy (S0)
    
    These estimates are based on regression relationships fitted to known data for organic CHO compounds.

    Parameters: 
        - formula : str
            Chemical formula of compound (e.g., "C6H10O5")
        - stdthermo : pd.DataFrame
            Standard Gibbs and enthalpy values for CO2, H2O and O2
        - elemental_entropies : pd.Series
            Entropies of the elements [C, H, O] in J/mol K
    
    Returns: 
        - pd.DataFrame : single-row DataFrame containing predicted thermodynamic properties
    """
    if isinstance(formulas, str): 
        formulas = [formulas]
    results = []

    for formula in formulas: 
        counts = parse_formulas(formula)
        a, b, c = int(counts.get("C", 0)), int(counts.get("H", 0)), int(counts.get("O", 0))
        MW = molar_mass(formula)
        
        #Stoichiometry per mole of compound
        vO2 = a + (b/4) - (c/2)
        if vO2 <= 0: 
            raise ValueError(f"vO2 =< 0 ({vO2:.3f}) for {formula}; too oxygen-rich, cannot combust")
        elif vO2 < 0.5: 
            warnings.warn(
                f"Compound {formula} has low oxygen demand (vO2 = {vO2:.2f}); predictions may be unreliable for oxygen-rich, low carbon species.", 
                UserWarning
            )
        
        #Predict dHC on model basis
        dHc_predict = theta0 + kO2*vO2
        dGc_predict = 137.88 + 411.21*a + 102.8*b - 205.6*c
        dHf_predict = -151.67 - 29.33*a + 37.21*b - 211.42*c 
        dGf_predict = 147.5 + 16.65*a - 15.77*b - 205.6*c 
        
        dSf_predict = ((dHf_predict - dGf_predict) / T0) * 1000 # J/mol K
        S0_predict = dSf_predict + (a * elemental_entropies.iloc[0] + b * elemental_entropies.iloc[1] + c * elemental_entropies.iloc[2])

        results.append({
            "Formula": formula, 
            "C": a, "H": b, "O": c, 
            "nu_O2": vO2, 
            "MW (g/mol)": MW,  
            "ΔH_C (kJ/mol)": dHc_predict, 
            "ΔG_C (kJ/mol)": dGc_predict,
            "ΔH_F (kJ/mol)": dHf_predict, 
            "ΔG_F (kJ/mol)": dGf_predict, 
            "ΔS_F (J/mol*K)": dSf_predict, 
            "S0 (J/mol*K))": S0_predict
        })
    
    
    return pd.DataFrame(results)

In [11]:
formulas = ["C3H7O2", "C6H12O6", "CH4", "C2H6O"]
pred_thermo = predict_org_thermo(formulas, stdthermo, elemental_entropies)
print(pred_thermo.to_string(index=False))

Formula  C  H  O  nu_O2  MW (g/mol)  ΔH_C (kJ/mol)  ΔG_C (kJ/mol)  ΔH_F (kJ/mol)  ΔG_F (kJ/mol)  ΔS_F (J/mol*K)  S0 (J/mol*K))
 C3H7O2  3  7  2   3.75    75.08658        1737.32        1679.91        -402.03        -324.14     -261.244340     418.784660
C6H12O6  6 12  6   6.00   180.15528        2688.71        2605.14       -1149.65       -1175.44       86.500084    1521.028084
    CH4  1  4  0   2.00    16.04276         997.35         960.29         -32.16         101.07     -446.855610    -179.662610
  C2H6O  2  6  1   3.00    46.06864        1420.19        1371.50        -198.49        -119.42     -265.202079     241.078921
